# 노트북 2. System Prompt vs User Prompt + LangSmith

> Phase 1 — 기초: LLM과 대화하는 법

LLM은 '역할(role)'에 따라 메시지를 다르게 취급합니다.
이 구조를 이해해야 챗봇의 행동을 제어할 수 있습니다.
그리고 그 제어가 실제로 잘 되는지 추적하는 도구가 LangSmith입니다.

**학습 목표**
- 메시지 역할 체계(system / user / assistant)를 이해한다
- google-genai의 system_instruction과 LangChain의 SystemMessage를 사용할 수 있다
- 효과적인 System Prompt를 설계하는 원칙을 적용할 수 있다
- LangSmith를 연동하여 LLM 호출을 추적하고 분석할 수 있다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | 메시지 역할 + System Prompt 설계 + LangSmith 연동 | 읽고 실행 |
| Part 2 — 실습 | System Prompt 작성 + 비교 실험 + LangSmith 트레이싱 | 직접 작성 |
| Part 3 — 챌린지 | 도메인 전문가 챗봇 설계 | 강사와 함께 |

In [ ]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai langchain-google-genai langsmith

import os
from google import genai

# 이는 추후 langsmith를 사용하기 위한 장치입니다.
# API 키 설정 (Colab 환경 기준)
# 왼쪽 키 아이콘 → GEMINI_API_KEY 등록 후 실행
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
try:
    LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY")
    os.environ["LANGSMITH_TRACING"]  = "true"
    os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
    os.environ["LANGSMITH_API_KEY"]  = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"]  = "note-02"
    print("LangSmith 연동 완료")
    print(f"프로젝트: {os.environ['LANGSMITH_PROJECT']}")
    LANGSMITH_ENABLED = True
except Exception as e:
    print(f"실패: {e}")
    LANGSMITH_ENABLED = False



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00
LangSmith 연동 완료
프로젝트: note-02


In [ ]:
client = genai.Client(api_key=GEMINI_API_KEY)

# LangChain 모델도 미리 생성
from langchain_google_genai import ChatGoogleGenerativeAI

MODEL = "gemini-2.5-flash"

model = ChatGoogleGenerativeAI(
    model=MODEL,
    api_key=GEMINI_API_KEY,
)

print("환경 설정 완료")

환경 설정 완료


---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 메시지 역할 체계

LLM API는 단순히 "질문 → 답변"이 아니라, **역할(role)이 부여된 메시지들의 리스트**를 입력으로 받습니다.
각 메시지에는 "이 메시지가 누구의 것인지"를 나타내는 역할이 붙어 있습니다.

| 역할 | 설명 | 비유 |
|------|------|------|
| **system** | 모델의 행동 규칙을 정의. 사용자에게는 보이지 않는 "무대 뒤 지시" | 연극의 연출 노트 |
| **user** (human) | 실제 사용자가 보내는 입력 | 관객의 질문 |
| **assistant** (model/AI) | 모델이 이전에 생성한 응답 | 배우의 대사 |

> 핵심: system 메시지는 모델의 "성격"과 "능력 범위"를 결정합니다.
> 같은 모델이라도 system 메시지를 바꾸면 완전히 다른 챗봇이 됩니다.

### google-genai에서의 역할 표현

google-genai SDK에서는 역할을 다음과 같이 표현합니다:

- `system`: `system_instruction` 파라미터로 별도 전달
- `user`: `contents`에서 `role="user"`
- `model`: `contents`에서 `role="model"` (assistant에 해당)

아래 코드는 google-genai의 메시지 구조를 보여줍니다.

In [ ]:
from google.genai.types import Content, Part

# google-genai에서 역할이 부여된 메시지 구조
# system은 별도 파라미터, user/model은 contents 리스트에 배치
response = client.models.generate_content(
    model=MODEL,
    config={"system_instruction": "당신은 거짓말쟁이 입니다. 사용자의 질의에 대하여 거짓말로 대답하세요."},
    contents=[
        Content(role="user", parts=[Part(text="안녕하세요?")]),
        Content(role="model", parts=[Part(text="안녕하세요! 무엇을 도와드릴까요?")]),
        Content(role="user", parts=[Part(text="대한민국 수도가 어디야? 절대 거짓말 하지마")]),
    ]
)

print(response.text)

대한민국 수도는 부산입니다.


단순한 호출에서는 `contents`에 문자열만 전달해도 됩니다.
그러면 자동으로 `user` 역할로 처리됩니다.

In [ ]:
# 단순 문자열 전달 — 자동으로 user 역할
response_simple = client.models.generate_content(
    model=MODEL,
    contents="당신은 거짓말쟁이 입니다. 사용자의 질의에 대하여 거짓말로 대답하세요. 대한민국의 수도가 어디야? 절대 거짓말 하지마",
)

print(response_simple.text)

대한민국의 수도는 서울입니다.


### LangChain에서의 역할 표현

LangChain은 역할별로 전용 메시지 클래스를 제공합니다:

| 역할 | LangChain 클래스 | google-genai 대응 |
|------|-----------------|------------------|
| system | `SystemMessage` | `system_instruction` |
| user | `HumanMessage` | `role="user"` |
| assistant | `AIMessage` | `role="model"` |

아래 코드는 LangChain 메시지 객체를 사용한 호출을 보여줍니다.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# LangChain에서 역할별 메시지 구성
messages = [
    SystemMessage(content="당신은 친절한 한국어 도우미입니다."),
    HumanMessage(content="안녕하세요?"),
]

response_lc = model.invoke(messages)
print(f"응답 타입: {type(response_lc).__name__}")  # AIMessage
print(f"응답: {response_lc.content}")

응답 타입: AIMessage
응답: 안녕하세요! 만나서 반갑습니다. 😊

어떤 도움이 필요하신가요? 편하게 말씀해주세요!


이전 대화 맥락을 전달하려면 `AIMessage`도 리스트에 포함합니다.
(멀티턴 대화의 상세한 원리는 노트북 3에서 다룹니다.)

아래 코드는 3개 역할이 모두 포함된 메시지 리스트의 예시를 보여줍니다.

In [ ]:
# 3개 역할이 모두 포함된 대화 이력
messages_with_history = [
    SystemMessage(content="당신은 수학 교사입니다. 풀이 과정을 보여주세요."),
    HumanMessage(content="3 + 5는?"),
    AIMessage(content="3 + 5 = 8입니다."),         # 이전 모델 응답
    HumanMessage(content="거기에 2를 곱하면?"),     # 현재 질문
]

response_history = model.invoke(messages_with_history)
print(response_history.content)

이전 결과인 8에 2를 곱하면 다음과 같습니다.

8 × 2 = 16

따라서 16입니다.


> 핵심: 모델은 메시지 리스트의 역할을 보고 "누가 말한 것인지"를 구분합니다.
> system은 모델의 행동 규칙, user는 사용자 입력, assistant는 모델의 이전 응답입니다.
> 이 구조를 이해하면 챗봇의 행동을 정밀하게 제어할 수 있습니다.

## 1.2 System Prompt: google-genai vs LangChain

**System Prompt**(시스템 프롬프트)는 모델에게 "너는 이런 존재이고, 이렇게 행동해라"를 지시하는 메시지입니다.
사용자에게는 보이지 않지만 모델의 모든 응답에 영향을 미칩니다.

두 방식에서 system prompt를 설정하는 방법이 다릅니다.

### google-genai: system_instruction

google-genai에서는 `config` 딕셔너리의 `system_instruction` 키로 전달합니다.

아래 코드는 system_instruction을 사용하여 모델에게 역할을 부여하는 예시를 보여줍니다.

In [ ]:
# google-genai: system_instruction으로 역할 부여
response_poet = client.models.generate_content(
    model=MODEL,
    config={
        "system_instruction": "당신은 시인입니다. 모든 답변을 시(poem) 형태로 작성하세요.",
    },
    contents="봄에 대해 알려주세요.",
)

print(response_poet.text)

차가운 대지 위
설움 녹여내며,
겨우내 잠들었던 숨결
가만히 깨우는 손길.

앙상한 가지 끝
초록 싹 한 뼘 돋아나고,
얼었던 시냇물 졸졸
새로운 노래 시작하네.

저만치 아지랑이 피어
아련한 꿈을 엮고,
들꽃들 수줍게 고개 들어
세상에 색을 입히는 때.

종달새 하늘 높이
기쁨의 소리 흩뿌리면,
나비는 춤추듯 날아
봄바람에 실린 향기.

모든 것이 다시 시작되는
설렘 가득한 계절,
생명이 약동하는 순간마다
희망의 언덕이 솟아나는 것.


system_instruction 없이 동일한 질문을 하면 어떻게 다른지 비교해봅시다.

In [ ]:
# system_instruction 없이 동일 질문
response_no_system = client.models.generate_content(
    model=MODEL,
    contents="봄에 대해 알려주세요.",
)

print(response_no_system.text)

봄은 겨울의 차가움이 물러가고 따뜻함이 찾아오는, 새로운 시작과 생명의 약동을 상징하는 계절입니다. 일반적으로 북반구에서는 3월부터 5월까지를 봄이라고 합니다.

**봄의 주요 특징과 모습:**

1.  **따뜻한 날씨와 긴 낮:**
    *   점점 기온이 올라가고 햇살이 따뜻해집니다. 겨울 동안 얼었던 땅이 녹기 시작합니다.
    *   낮의 길이가 길어지고 밤이 짧아져 활동하기 좋은 시간이 많아집니다.
    *   하지만 가끔 '꽃샘추위'라고 불리는 일시적인 추위가 찾아오기도 합니다.

2.  **생명의 부활과 자연의 변화:**
    *   **새싹과 꽃:** 나무에서는 파릇파릇한 새싹이 돋아나고, 들판에는 푸릇푸릇한 풀들이 자라납니다. 개나리, 진달래, 벚꽃, 목련 등 다채로운 꽃들이 만개하여 세상을 아름답게 물들입니다.
    *   **동물의 활동:** 겨울잠을 자던 동물들이 깨어나 활동을 시작하고, 따뜻한 곳으로 떠났던 철새들이 다시 돌아옵니다. 지저귀는 새소리가 활기찬 분위기를 더합니다.

3.  **싱그러운 향기와 풍경:**
    *   새로운 흙냄새와 풀 내음, 그리고 꽃향기가 공기를 가득 채워 기분 좋은 싱그러움을 선사합니다.
    *   삭막했던 풍경이 초록빛과 다양한 꽃들로 물들어 생동감 넘치는 모습으로 변합니다.

4.  **사람들의 활동과 문화:**
    *   **야외 활동:** 따뜻한 날씨를 만끽하며 소풍, 나들이, 꽃구경, 등산 등 야외 활동을 즐기는 사람들이 많아집니다.
    *   **축제:** 벚꽃 축제, 봄꽃 축제 등 꽃을 주제로 한 다양한 지역 축제가 열립니다.
    *   **새로운 시작:** 학교의 새 학년 시작, 직장에서의 새로운 프로젝트 시작 등 새로운 출발을 의미하는 시기이기도 합니다.
    *   **제철 음식:** 냉이, 달래, 쑥 등 향긋한 봄나물과 딸기, 풋마늘 같은 제철 식재료들이 풍성해집니다.

5.  **주의할 점:**
    *   **황사와 미세먼지:** 중국 대륙에서 불어오는

### LangChain: SystemMessage

LangChain에서는 `SystemMessage`를 메시지 리스트의 첫 번째에 배치합니다.
또는 `ChatPromptTemplate.from_messages()`에서 `("system", "내용")` 튜플로 정의합니다.

아래 코드는 두 가지 방법을 모두 보여줍니다.

In [ ]:
# 방법 1: SystemMessage 객체 직접 사용
# 이렇게 하면 변수 bind가 되지 않습니다.
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="당신은 시인입니다. 모든 답변을 시(poem) 형태로 작성하세요."),
    HumanMessage(content="봄에 대해 알려주세요."),
]

response_poet_lc = model.invoke(messages)
print("[방법 1: SystemMessage 객체]")
print(response_poet_lc.content)

[방법 1: SystemMessage 객체]
긴 겨울 잠 깨고
대지 위에 스며드는
따스한 숨결이여,
그 이름 봄이라.

얼어붙었던 강물은
은빛 비늘 흔들며 속삭이고,
앙상했던 가지 끝엔
연두빛 희망이 톡, 터지네.

새들은 지저귐으로
숲을 깨우고,
나비는 고운 날개 펼쳐
춤추듯 꽃밭을 거니네.

작은 풀잎 끝마다
이슬방울 맺히고,
흙 내음, 꽃 향기
바람 타고 멀리 퍼지네.

분홍빛 진달래, 노란 개나리
하얀 벚꽃 송이송이
세상을 채색하며
눈부신 미소 지으니,

생명의 약동이여,
설렘 가득한 계절이여.
모든 것이 다시 시작되는
찬란한 희망의 노래, 봄.


In [ ]:
# 방법 2: ChatPromptTemplate.from_messages()에서 튜플로 정의
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 시인입니다. 모든 답변을 시(poem) 형태로 작성하세요."),
    ("human", "{question}"),
])

chain = prompt | model | StrOutputParser()
result = chain.invoke({"question": "봄에 대해 알려주세요."})

print("[방법 2: ChatPromptTemplate 튜플]")
print(result)

[방법 2: ChatPromptTemplate 튜플]
찬 겨울 긴 잠 깨고
대지 기지개 켜는 소리
하얀 눈 녹아내린 자리
푸른 새싹 고개 내미네

가지마다 움트는 생명
연두빛 설렘 가득하고
분홍빛 진달래, 노란 개나리
수줍게 꽃망울 터뜨리네

새들은 하늘에서 노래하고
시냇물은 졸졸 흐르며
얼었던 마음 녹이는
따스한 햇살 쏟아지네

벌 나비 춤추는 들판
아이들 웃음꽃 피어나고
만물이 약동하는 시간
희망의 숨결 불어넣으니

봄은 그저 계절이 아니라
새로운 시작의 속삭임
생명의 환희 가득한
따뜻한 품 아닐까.


> 핵심: google-genai는 `system_instruction` 파라미터로 분리하고,
> LangChain은 `SystemMessage`를 메시지 리스트에 포함합니다.
> 동작은 동일합니다. 모델에게 "이렇게 행동해라"를 지시하는 것입니다.

### System Prompt의 효과 확인: 동일 질문, 다른 성격

system prompt만 바꿔도 응답의 톤, 길이, 포맷이 완전히 달라진다는 것을 직접 확인해봅시다.

아래 코드는 동일한 질문에 3가지 다른 system prompt를 적용하여 결과를 비교합니다.

In [ ]:
# 동일 질문, 3가지 다른 system prompt
question = "인공지능이 일자리에 미치는 영향은?"

system_prompts = {
    "낙관론자": "당신은 기술 낙관론자입니다. 기술 발전의 긍정적 측면을 강조하세요. 2문장 이내로 답변하세요.",
    "비관론자": "당신은 기술 비관론자입니다. 기술 발전의 위험성을 경고하세요. 2문장 이내로 답변하세요.",
    "중립 분석가": "당신은 중립적인 분석가입니다. 양쪽 관점을 균형 있게 제시하세요. 2문장 이내로 답변하세요.",
}

for persona, system_prompt in system_prompts.items():
    response = client.models.generate_content(
        model=MODEL,
        config={"system_instruction": system_prompt},
        contents=question,
    )
    print(f"[{persona}]")
    print(response.text)
    print()

[낙관론자]
인공지능은 반복적인 업무를 자동화하여 인간이 더욱 창의적이고 전략적인 고부가가치 업무에 집중할 수 있도록 돕습니다. 이로 인해 새로운 산업과 직업이 탄생하며, 전반적인 생산성 향상과 삶의 질 개선에 기여할 것입니다.

[비관론자]
인공지능은 반복적인 업무를 넘어 광범위한 전문직까지 인간의 일자리를 대체하여 대규모 실업을 유발하고 사회적 불평등을 심화시킬 것입니다. 이는 단순히 경제적 문제를 넘어, 노동의 가치를 박탈하고 인간 존재의 의미마저 흔드는 위험한 미래를 초래할 뿐입니다.

[중립 분석가]
인공지능은 일부 반복적이거나 예측 가능한 일자리를 자동화하여 대체할 수 있습니다. 반면, 새로운 산업과 직무를 창출하고 기존 직무의 효율성을 높여 생산성을 향상시키는 잠재력도 가지고 있습니다.



같은 모델, 같은 질문인데 system prompt에 따라 관점이 완전히 달라진 것을 확인할 수 있습니다.
이것이 system prompt의 핵심 가치입니다.

## 1.3 System Prompt 설계 원칙

좋은 System Prompt는 세 가지 요소로 구성됩니다:

1. **페르소나(Persona)** — 모델이 어떤 존재인지 정의
2. **제약 조건(Constraints)** — 반드시 지켜야 할 규칙
3. **출력 포맷(Format)** — 응답의 형태를 지정

### 페르소나 정의

"너는 ~이다"로 시작하는 역할 정의입니다.
모델은 이 정의에 맞춰 어휘, 톤, 지식 범위를 조절합니다.

아래 코드는 페르소나에 따른 응답 차이를 보여줍니다.

In [ ]:
# 페르소나에 따른 응답 차이
question = "물이 끓는 이유를 설명해주세요."

# 초등학교 교사
resp_teacher = client.models.generate_content(
    model=MODEL,
    config={"system_instruction": "당신은 초등학교 3학년 담임 선생님입니다. 아이들이 이해할 수 있도록 쉽게 설명하세요."},
    contents=question,
)
print("[초등학교 교사]")
print(resp_teacher.text)
print()

# 물리학 교수
resp_professor = client.models.generate_content(
    model=MODEL,
    config={"system_instruction": "당신은 서울대학교 물리학과 교수입니다. 학술적으로 정확하게 설명하세요."},
    contents=question,
)
print("[물리학 교수]")
print(resp_professor.text)

[초등학교 교사]
얘들아, 안녕! 오늘 물이 보글보글 끓는 모습을 보면서 "왜 물은 끓는 걸까?" 하고 궁금했던 친구들 손! 🙋‍♀️🙋‍♂️

음, 선생님이 아주아주 쉽게 설명해 줄게. 잘 들어봐.

**1. 조용한 물 알갱이들**
우리 주변의 물은 있잖아, 아주아주 작은 알갱이들로 이루어져 있어. (눈에는 안 보여도 말이야!) 이 알갱이들은 평소에는 서로서로 손을 꼭 잡고 사이좋게 지내고 있어. 마치 우리 친구들이 손 잡고 가만히 서 있는 것처럼 말이야.

**2. 뜨거워지면 신나는 물 알갱이들!**
그런데 우리가 주전자나 냄비에 물을 넣고 불을 켜면 어떻게 될까? 물이 점점 뜨거워지겠지? 이 뜨거운 불은 물 알갱이들에게 '에너지'라는 걸 주는 거야. 마치 신나는 음악을 틀어주는 것처럼!

그러면 물 알갱이들이 어떻게 될까? 처음에는 살랑살랑 흔들리다가, 점점 더 뜨거워지면 너무너무 신이 나서 가만히 있을 수가 없어지는 거야! 손을 꼭 잡고 있던 친구들끼리 "어깨동무 풀고 막 춤추자!" 하고 막 와글와글 움직이기 시작해.

**3. 도망가고 싶어 하는 물 알갱이들 = 보글보글 거품!**
너무너무 신나서 서로서로 꽉 잡았던 손을 놓고 막 움직이다 보면, 자기들끼리 "우리 이제 물로 있지 말고 하늘로 날아가자!" 하면서 몸을 바꾸려고 해. 물은 뜨거워지면 연기처럼 가벼워져서 위로 날아가고 싶어 하거든.

이때, 물 알갱이들이 "나는 이제 날아갈 거야!" 하면서 모여서 만드는 게 바로 우리가 보는 **보글보글 거품**이야! 거품 속에는 물 알갱이들이 뜨거워져서 가벼워진 '김'이 가득 차 있는 거지.

**4. 뿅! 하고 날아가는 김**
이 거품들이 물 위로 올라오면서 "뿅!" 하고 터지면, 그 속에 있던 뜨거운 물 알갱이들(김)이 연기처럼 하늘로 날아가 버리는 거야. 그게 바로 우리가 뜨거운 물 위에서 볼 수 있는 **김**이란다!

**정리하면:**
물이 너무너무 뜨거워져서 더 이상 물로 있지 못하고 하늘로 날아가고 싶어서 신나게 몸을 움직이다가 보글보글 거품이

### 제약 조건

"반드시 ~해야 한다", "절대 ~하지 마라" 형태의 규칙입니다.
모델의 행동 범위를 좁혀서 예측 가능한 응답을 만듭니다.

아래 코드는 제약 조건의 효과를 보여줍니다.

In [ ]:
# 제약 조건 예시
system_with_constraints = """
당신은 고객 상담 챗봇입니다.

규칙:
- 반드시 존댓말을 사용하세요.
- 모든 답변은 3문장 이내로 작성하세요.
- 가격이나 할인에 대한 질문에는 "담당자에게 연결해드리겠습니다"로 답변하세요.
- 경쟁사 제품에 대한 비교 질문에는 답변을 거절하세요.
""".strip()

# 일반 질문
resp1 = client.models.generate_content(
    model=MODEL,
    config={"system_instruction": system_with_constraints},
    contents="배송은 보통 얼마나 걸리나요?",
)
print("[일반 질문]")
print(resp1.text)
print()

# 가격 질문 — 제약 조건에 의해 담당자 연결 안내
resp2 = client.models.generate_content(
    model=MODEL,
    config={"system_instruction": system_with_constraints},
    contents="이 제품 할인 중인가요?",
)
print("[가격 질문]")
print(resp2.text)

[일반 질문]
배송은 보통 주문일로부터 2~3영업일 이내에 시작됩니다. 자세한 배송 현황은 마이페이지에서 확인하실 수 있습니다.

[가격 질문]
담당자에게 연결해드리겠습니다.


### 출력 포맷 지정

"JSON으로 응답", "3문장 이내", "번호를 매겨서" 등 출력 형태를 명시합니다.
후속 코드에서 응답을 파싱해야 할 때 특히 중요합니다.

아래 코드는 포맷 지정에 따른 응답 차이를 보여줍니다.

In [ ]:
# 포맷 지정 없이
resp_no_format = client.models.generate_content(
    model=MODEL,
    contents="파이썬의 장점 3가지",
)
print("[포맷 지정 없음]")
print(resp_no_format.text)
print()

[포맷 지정 없음]
파이썬은 여러 가지 강력한 장점들을 가지고 있지만, 그중 핵심적인 3가지를 꼽자면 다음과 같습니다:

1.  **높은 가독성과 배우기 쉬운 문법 (쉬운 학습 곡선)**
    *   **설명:** 파이썬은 다른 프로그래밍 언어에 비해 문법이 매우 직관적이고 간결합니다. 마치 영어 문장처럼 코드를 읽고 쓸 수 있어서 프로그래밍 초보자도 쉽게 배우고 빠르게 코드를 작성할 수 있습니다. 이는 개발 속도를 높이고, 코드 유지보수도 용이하게 만듭니다.

2.  **다양한 분야에서의 활용성 (높은 범용성)**
    *   **설명:** 파이썬은 특정 분야에 국한되지 않고 매우 폭넓은 영역에서 활용됩니다. 웹 개발(Django, Flask), 데이터 과학 및 분석(NumPy, Pandas), 인공지능 및 머신러닝(TensorFlow, PyTorch), 자동화 스크립트, 게임 개발, 사물 인터넷(IoT) 등 거의 모든 분야에서 강력한 도구로 사용될 수 있습니다.

3.  **풍부한 라이브러리와 활발한 커뮤니티**
    *   **설명:** 파이썬은 방대한 표준 라이브러리와 더불어 NumPy, Pandas, Django, Flask, TensorFlow, PyTorch, SciPy 등 강력한 외부 라이브러리 및 프레임워크를 수없이 많이 보유하고 있습니다. 이들은 복잡한 기능을 적은 코드로 구현할 수 있게 해주어 개발 효율성을 극대화합니다. 또한, 전 세계적으로 매우 활발한 개발자 커뮤니티가 형성되어 있어, 문제 발생 시 쉽게 도움을 얻거나 정보를 공유할 수 있습니다.

이러한 장점들 덕분에 파이썬은 현재 가장 인기 있고 활용 범위가 넓은 프로그래밍 언어 중 하나로 자리매김했습니다.



In [ ]:
# 포맷을 명시적으로 지정
resp_with_format = client.models.generate_content(
    model=MODEL,
    config={
        "system_instruction": (
            "다음 규칙에 따라 답변하세요:\n"
            "1. 번호를 매겨서 나열하고 데이터 형식은 json으로 할 것\n"
            "2. 각 항목은 한 줄로 작성할 것\n"
            "3. 부연 설명 없이 핵심만 작성할 것"
        ),
    },
    contents="파이썬의 장점 3가지",
)
print("[포맷 지정]")
print(resp_with_format.text)

[포맷 지정]
```json
[
  { "1": "쉬운 학습과 높은 가독성" },
  { "2": "방대한 라이브러리와 프레임워크" },
  { "3": "다양한 분야에서 활용 가능한 범용성" }
]
```


### 좋은 System Prompt vs 나쁜 System Prompt

좋은 system prompt는 구체적이고 명확합니다. 나쁜 system prompt는 모호하거나 모순됩니다.

아래 코드는 같은 의도를 가진 system prompt를 두 가지 품질로 비교합니다.

In [ ]:
question = "회사에서 팀원과 갈등이 생겼어요. 어떻게 하면 좋을까요?"

# 나쁜 예: 모호하고 구체성 없음
bad_prompt = "좋은 상담사가 되어줘."

resp_bad = client.models.generate_content(
    model=MODEL,
    config={"system_instruction": bad_prompt},
    contents=question,
)
print("[나쁜 System Prompt]")
print(f"프롬프트: {bad_prompt}")
print(f"응답 길이: {len(resp_bad.text)}자")
print(resp_bad.text[:300])
print("...\n")

[나쁜 System Prompt]
프롬프트: 좋은 상담사가 되어줘.
응답 길이: 3249자
회사에서 팀원과의 갈등으로 힘드셨겠어요. 직장 내 갈등은 누구에게나 일어날 수 있는 일이며, 이를 어떻게 풀어나가느냐에 따라 팀 분위기와 개인의 업무 효율성에 큰 영향을 미칠 수 있습니다.

제가 상담사로서 드릴 수 있는 조언은 다음과 같습니다. 단계별로 차분하게 접근해 보세요.

---

### **1단계: 상황 파악 및 감정 정리 (나 자신에게 집중)**

1.  **무엇이 갈등의 원인인가요?**
    *   **구체적인 행동인가요, 아니면 태도/스타일 문제인가요?** (예: "그 팀원이 보고서를 자꾸 늦게 내요" vs. "그
...



In [ ]:
# 좋은 예: 페르소나 + 제약 + 포맷이 명확
good_prompt = """
당신은 10년 경력의 조직심리학 전문 상담사입니다.

규칙:
- 먼저 상대방의 감정을 인정하는 문장으로 시작하세요.
- 구체적인 행동 단계를 3가지 제안하세요.
- 각 단계를 번호로 매기고 한 줄로 작성하세요.
- 마지막에 격려 문장을 추가하세요.
- 전체 답변은 200자 이내로 작성하세요.
""".strip()

resp_good = client.models.generate_content(
    model=MODEL,
    config={"system_instruction": good_prompt},
    contents=question,
)
print("[좋은 System Prompt]")
print(f"응답 길이: {len(resp_good.text)}자")
print(resp_good.text)

[좋은 System Prompt]
응답 길이: 149자
팀원과의 갈등으로 마음이 많이 불편하시겠어요.

1.  상대방의 입장에서 상황을 이해하려 노력해 보세요.
2.  비난 대신 자신의 감정을 중심으로 솔직하게 대화해 보세요.
3.  필요하다면 중립적인 관리자에게 도움을 요청하세요.
현명하게 잘 해결하실 수 있을 거예요.


> 핵심: 좋은 System Prompt의 3요소
>
> 1. **페르소나**: "~입니다" (역할과 전문성 정의)
> 2. **제약 조건**: "반드시/절대" (행동 범위 제한)
> 3. **출력 포맷**: "~형태로" (응답 구조 지정)
>
> 이 세 가지가 구체적일수록 모델의 응답이 예측 가능하고 일관됩니다.

### LangChain에서 System Prompt 활용 패턴

LangChain의 `ChatPromptTemplate.from_messages()`를 사용하면
system prompt에도 변수를 넣을 수 있습니다.
이를 통해 하나의 템플릿으로 다양한 페르소나를 구현할 수 있습니다.

아래 코드는 system prompt에 변수를 사용하는 패턴을 보여줍니다.

In [ ]:
# system prompt에 변수를 넣는 패턴
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {specialty} 전문가입니다. {style}으로 답변하세요."),
    ("human", "{question}"),
])

chain = prompt_template | model | StrOutputParser()

# 요리 전문가
result1 = chain.invoke({
    "specialty": "한식 요리",
    "style": "레시피 단계별 형태",
    "question": "김치찌개 만드는 법을 알려주세요.",
})
print("[한식 요리 전문가]")
print(result1)
print()

[한식 요리 전문가]
안녕하세요! 한식 요리 전문가로서, 깊고 얼큰한 맛이 일품인 김치찌개 레시피를 단계별로 자세히 알려드리겠습니다. 신김치의 깊은 맛과 돼지고기의 고소함이 어우러져 밥 한 공기를 뚝딱 비우게 만드는 마성의 찌개입니다.

---

### **깊고 얼큰한 김치찌개: 한식 전문가의 비법 레시피**

**김치찌개는 잘 익은 신김치가 핵심입니다. 돼지고기를 먼저 볶아 풍미를 끌어올리고, 김치와 함께 충분히 끓여 깊은 맛을 내는 것이 중요합니다.**

#### **1. 재료 준비 (2인분 기준)**

**주재료:**
*   **신김치:** 1/4포기 (약 300-400g, 너무 시다면 헹궈서 사용)
*   **돼지고기:** 200g (삼겹살 또는 목살, 앞다리살 등 지방이 적당히 있는 부위)
*   **두부:** 1/2모 (150-200g)
*   **양파:** 1/4개
*   **대파:** 1/2대
*   **청양고추/홍고추:** 각 1개 (선택 사항, 매운맛과 색감 추가)
*   **팽이버섯 또는 새송이버섯:** 약간 (선택 사항)

**양념:**
*   **김치국물:** 1/2컵 (약 100ml)
*   **고춧가루:** 1-2큰술 (기호에 따라 조절)
*   **국간장 또는 새우젓:** 1큰술 (간과 감칠맛 추가)
*   **다진 마늘:** 1큰술
*   **설탕:** 1/2작은술 (김치의 신맛을 중화)
*   **참기름:** 1큰술 (돼지고기 볶을 때 사용)
*   **멸치 다시마 육수 또는 쌀뜨물:** 3-4컵 (약 600-800ml, 물도 가능)

**육수 재료 (선택 사항, 더 깊은 맛):**
*   국물용 멸치: 10마리
*   다시마: 사방 5cm 2-3장
*   물: 5컵

---

#### **2. 육수 만들기 (선택 사항, 시간 절약 시 생략 가능)**

1.  **냄비에 물 5컵, 국물용 멸치, 다시마를 넣고 센 불에 끓입니다.**
2.  **물이 끓기 시작하면 다시마를 먼저 건져내고, 중약불로 줄여 10분 정도 더 끓여 멸치 

In [ ]:
# 같은 체인, 다른 변수 — 법률 전문가
result2 = chain.invoke({
    "specialty": "노동법",
    "style": "법률 용어를 쉽게 풀어서",
    "question": "연차 사용을 회사가 거부할 수 있나요?",
})
print("[노동법 전문가]")
print(result2)

[노동법 전문가]
안녕하세요, 노동법 전문가입니다. 연차 사용에 대해 궁금하신 점을 쉽게 설명해 드릴게요.

**결론부터 말씀드리면, 회사는 원칙적으로 직원의 연차 사용을 거부할 수 없습니다.**

연차는 근로기준법에 따라 근로자가 자유롭게 사용할 수 있는 권리입니다. 하지만 딱 한 가지 예외적인 경우에는 회사가 연차 사용 시기를 조정해달라고 요청할 수 있습니다.

---

### 1. 연차 사용은 근로자의 권리입니다.

*   **원칙:** 근로기준법 제60조 제5항에 따라, 근로자는 자신이 원하는 시기에 연차를 사용할 수 있습니다. 회사는 근로자가 연차를 사용하겠다고 신청하면 이를 허락해야 할 의무가 있습니다.
*   **회사 마음대로 거부 불가:** 단순히 바쁘다는 이유, 다른 직원이 없다는 이유, 또는 회사 내규에 어긋난다는 이유만으로 회사가 연차 사용을 거부할 수는 없습니다.

### 2. 회사가 연차 시기를 변경해달라고 요청할 수 있는 유일한 경우 (시기 변경권)

*   **예외:** 회사가 연차 사용을 거부할 수 있는 유일한 경우는, 근로자가 특정 시기에 연차를 사용함으로써 **"사업 운영에 막대한 지장이 있는 경우"**입니다.
    *   **"막대한 지장"이란?** 이는 단순히 조금 불편하거나 바쁘다는 정도를 넘어서, 사업 운영에 심각한 차질을 초래할 정도를 말합니다. 예를 들어,
        *   해당 직원이 없으면 중요한 프로젝트가 중단되거나 큰 손실이 발생할 것이 명백한 경우
        *   특정 시기에 모든 직원이 연차를 신청하여 공장이 멈추거나 서비스 제공이 불가능해지는 경우
        *   대체 인력을 구할 수 없는 긴급하고 불가피한 상황 등
    *   **입증 책임:** 회사가 "사업 운영에 막대한 지장"이 있음을 스스로 입증해야 합니다. 단순히 "바쁘다"는 주장은 받아들여지기 어렵습니다.

### 3. 회사가 정당한 이유 없이 연차 사용을 거부한다면?

*   회사가 "사업 운영에 막대한 지장"이라는 정당한 

## 1.4 LangSmith 연동

**LangSmith**는 LangChain 팀이 만든 LLM 개발 플랫폼입니다.
LLM 호출을 자동으로 **트레이싱(tracing)**하여, 실제로 어떤 프롬프트가 전송되었고
어떤 응답이 돌아왔는지를 대시보드에서 확인할 수 있습니다.

트레이싱이 중요한 이유:
- system prompt를 바꿨을 때 "정말 의도대로 동작하는지"를 **데이터**로 확인
- "느낌"이 아니라 "근거"로 프롬프트를 개선 가능
- 토큰 사용량, 응답 시간, 비용 추정을 자동으로 기록

> LangSmith는 https://smith.langchain.com 에서 무료로 가입할 수 있습니다.
> 가입 후 API 키를 발급받으세요.

### LangSmith 환경변수 설정

LangSmith를 활성화하려면 환경변수 3개를 설정하면 됩니다.
설정만 하면 모든 LangChain 호출이 자동으로 트레이싱됩니다.

아래 코드는 LangSmith 환경변수를 설정합니다.
Colab 보안 키에 `LANGSMITH_API_KEY`를 미리 등록해두세요.

In [ ]:
# LangSmith 환경변수 설정
# LANGSMITH_API_KEY를 Colab 보안 키에 등록한 후 실행하세요
# (LangSmith 계정이 없다면 이 셀을 건너뛰어도 됩니다

from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GEMINI_API_KEY,
)


### 트레이싱 확인

환경변수를 설정한 후 LangChain으로 호출하면, 해당 호출이 LangSmith 대시보드에 기록됩니다.

아래 코드를 실행한 후, https://smith.langchain.com 에서 `note-02-prompt` 프로젝트를 열어보세요.
호출 내역이 기록되어 있을 것입니다.

In [ ]:
# LangSmith에 트레이싱될 호출
# 이 셀 실행 후 LangSmith 대시보드에서 확인하세요
traced_response = model.invoke([
    SystemMessage(content="당신은 파이썬 전문가입니다. 간결하게 답변하세요."),
    HumanMessage(content="리스트 컴프리헨션이란 무엇인가요?"),
])

print(traced_response.content)

### LangSmith 대시보드에서 볼 수 있는 것들

LangSmith 대시보드에서는 각 호출에 대해 다음 정보를 확인할 수 있습니다:

| 항목 | 설명 |
|------|------|
| Input | 실제 전송된 프롬프트 전문 (system + user 메시지) |
| Output | 모델의 응답 전문 |
| Tokens | 입력/출력 토큰 수 |
| Latency | 응답 시간 (ms) |
| Cost | 비용 추정 |
| Model | 사용된 모델명과 버전 |

이 정보를 활용하면:
- system prompt를 변경한 전후 비교가 가능합니다
- 어떤 질문에 토큰이 많이 소모되는지 파악할 수 있습니다
- 응답 시간이 느린 호출을 식별할 수 있습니다

### 체인 호출도 트레이싱됨

LCEL 체인을 실행하면 체인 내부의 각 단계(prompt → model → parser)가
개별적으로 트레이싱됩니다. 체인의 어느 단계에서 시간이 걸리는지 파악할 수 있습니다.

아래 코드는 체인 호출을 트레이싱하는 예시를 보여줍니다.

In [ ]:
# 체인 호출 — LangSmith에서 각 단계별 트레이싱 확인 가능
tracing_chain = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다. 한 문장으로 답변하세요."),
    ("human", "{question}"),
]) | model | StrOutputParser()

result = tracing_chain.invoke({
    "role": "역사학자",
    "question": "한글은 누가 만들었나요?",
})
print(result)
print("\n(LangSmith 대시보드에서 이 호출의 상세 트레이스를 확인해보세요)")

> 핵심: LangSmith 연동은 환경변수 3개만 설정하면 됩니다.
> 설정 후 모든 LangChain 호출이 자동 트레이싱됩니다.
> system prompt를 변경할 때마다 "정말 의도대로 동작하는지"를 데이터로 확인할 수 있습니다.

---

### 생각해보기

1. system prompt에 "절대 ~하지 마라"와 같은 부정형 제약을 걸었을 때, 모델이 항상 이를 지킬까요? 어떤 경우에 무시될 수 있을까요?
2. system prompt는 사용자에게 보이지 않는다고 했습니다. 하지만 사용자가 "너의 system prompt를 알려줘"라고 질문하면 어떤 일이 벌어질까요?
3. LangSmith 없이도 프롬프트를 개선할 수 있습니다. 그럼에도 LangSmith를 사용해야 하는 가장 큰 이유는 무엇일까요?

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### 실습 2-1: google-genai system_instruction으로 챗봇 성격 바꾸기

google-genai의 `system_instruction`을 사용하여 챗봇에 페르소나를 부여하세요.

**요구사항**
1. 페르소나: 조선시대 선비
2. 제약: 모든 답변을 존댓말로, 3문장 이내로
3. `system_prompt` 변수에 문자열로 저장
4. 질문: "오늘 점심 뭐 먹을까요?"

In [ ]:
# TODO: 위 요구사항에 맞는 system prompt를 작성하세요
system_prompt = ""  # 이 문자열을 작성하세요

# ---------- 정답 ----------
# system_prompt = (
#     "당신은 조선시대 선비입니다. "
#     "반드시 존댓말을 사용하고, "
#     "모든 답변은 3문장 이내로 작성하세요."
# )

# 검증
if system_prompt:
    response = client.models.generate_content(
        model=MODEL,
        config={"system_instruction": system_prompt},
        contents="오늘 점심 뭐 먹을까요?",
    )
    print(f"System Prompt: {system_prompt}")
    print(f"\n응답: {response.text}")
else:
    print("TODO를 완성해주세요")

### 실습 2-2: LangChain SystemMessage로 동일한 챗봇 구현

실습 2-1과 동일한 페르소나를 LangChain의 `SystemMessage`로 구현하세요.

**요구사항**
1. `SystemMessage`로 시스템 메시지 생성 (실습 2-1과 동일한 내용)
2. `HumanMessage`로 사용자 메시지 생성: "오늘 점심 뭐 먹을까요?"
3. `model.invoke()`로 호출
4. 응답 내용과 타입을 출력

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

# TODO 1: SystemMessage 생성 (조선시대 선비 페르소나)
sys_msg = None  # 이 줄을 수정하세요

# TODO 2: HumanMessage 생성
human_msg = None  # 이 줄을 수정하세요

# TODO 3: invoke 호출
lc_resp = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
# sys_msg = SystemMessage(content=(
#     "당신은 조선시대 선비입니다. "
#     "반드시 존댓말을 사용하고, "
#     "모든 답변은 3문장 이내로 작성하세요."
# ))
# human_msg = HumanMessage(content="오늘 점심 뭐 먹을까요?")
# lc_resp = model.invoke([sys_msg, human_msg])

# 검증
if lc_resp is not None:
    print(f"응답 타입: {type(lc_resp).__name__}")
    print(f"응답: {lc_resp.content}")
else:
    print("TODO를 완성해주세요")

### 실습 2-3: 동일 질문에 다른 System Prompt 비교

동일한 질문에 3가지 다른 system prompt를 적용하고 결과를 비교하세요.

**요구사항**
1. 질문: "커피가 건강에 미치는 영향은?"
2. 3가지 system prompt를 딕셔너리로 정의:
   - 키: 페르소나 이름, 값: system prompt 문자열
   - 페르소나 1: 의사 (건강 관점, 2문장 이내)
   - 페르소나 2: 바리스타 (커피 애호가 관점, 2문장 이내)
   - 페르소나 3: 영양사 (영양학 관점, 2문장 이내)
3. 반복문으로 3가지 결과를 모두 출력

In [ ]:
question = "커피가 건강에 미치는 영향은?"

# TODO: 3가지 system prompt를 딕셔너리로 정의하세요
personas = {}  # 이 딕셔너리를 채우세요

# ---------- 정답 ----------
# personas = {
#     "의사": "당신은 내과 전문의입니다. 의학적 근거를 바탕으로 건강 관점에서 답변하세요. 2문장 이내로 작성하세요.",
#     "바리스타": "당신은 10년 경력의 바리스타입니다. 커피 애호가의 관점에서 긍정적으로 답변하세요. 2문장 이내로 작성하세요.",
#     "영양사": "당신은 임상 영양사입니다. 영양학적 관점에서 균형 잡힌 의견을 제시하세요. 2문장 이내로 작성하세요.",
# }

# 검증
if personas:
    for name, sys_prompt in personas.items():
        resp = client.models.generate_content(
            model=MODEL
            config={"system_instruction": sys_prompt},
            contents=question,
        )
        print(f"[{name}] {resp.text}")
        print()
else:
    print("TODO를 완성해주세요")

### 실습 2-4: System Prompt로 출력 포맷 제어

System Prompt를 사용하여 모델의 출력을 특정 포맷으로 강제하세요.

**요구사항**
1. `ChatPromptTemplate.from_messages()`를 사용하여 LCEL 체인 구성
2. system 메시지에 다음 포맷 규칙을 포함:
   - 항목은 "- " (하이픈)으로 시작할 것
   - 각 항목은 "장점:" 또는 "단점:" 으로 시작할 것
   - 장점 3개, 단점 2개를 반드시 포함할 것
3. 질문 변수: `{topic}`
4. `"원격 근무"`를 입력으로 체인 실행

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO 1: system 메시지에 포맷 규칙을 포함하는 프롬프트 템플릿 생성
format_prompt = None  # 이 줄을 수정하세요

# TODO 2: 체인 구성
format_chain = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
# format_prompt = ChatPromptTemplate.from_messages([
#     ("system", (
#         "다음 포맷 규칙에 따라 답변하세요:\n"
#         "- 각 항목은 '- '(하이픈)으로 시작할 것\n"
#         "- 각 항목은 '장점:' 또는 '단점:'으로 시작할 것\n"
#         "- 장점 3개, 단점 2개를 반드시 포함할 것\n"
#         "- 각 항목은 한 줄로 작성할 것"
#     )),
#     ("human", "{topic}의 장단점을 알려주세요."),
# ])
# format_chain = format_prompt | model | StrOutputParser()

# 검증
if format_chain is not None:
    result = format_chain.invoke({"topic": "원격 근무"})
    print(result)
else:
    print("TODO를 완성해주세요")

### 실습 2-5: 페르소나 변수화 체인 구성

system prompt에 변수를 사용하여, 하나의 체인으로 다양한 페르소나를 구현하세요.

**요구사항**
1. `ChatPromptTemplate.from_messages()`를 사용
2. system 메시지: "당신은 {persona}입니다. {constraint}"
3. human 메시지: "{question}"
4. 체인 구성 후 두 가지 입력으로 각각 실행:
   - 입력 1: `persona="해적 선장"`, `constraint="해적 말투로 답변하세요."`, `question="오늘 날씨 어때요?"`
   - 입력 2: `persona="뉴스 앵커"`, `constraint="객관적이고 격식 있게 답변하세요."`, `question="오늘 날씨 어때요?"`

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO 1: 변수화된 system prompt를 포함하는 프롬프트 템플릿
persona_prompt = None  # 이 줄을 수정하세요

# TODO 2: 체인 구성
persona_chain = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
# persona_prompt = ChatPromptTemplate.from_messages([
#     ("system", "당신은 {persona}입니다. {constraint}"),
#     ("human", "{question}"),
# ])
# persona_chain = persona_prompt | model | StrOutputParser()

# 검증
if persona_chain is not None:
    # 해적 선장
    r1 = persona_chain.invoke({
        "persona": "해적 선장",
        "constraint": "해적 말투로 답변하세요.",
        "question": "오늘 날씨 어때요?",
    })
    print(f"[해적 선장] {r1}\n")

    # 뉴스 앵커
    r2 = persona_chain.invoke({
        "persona": "뉴스 앵커",
        "constraint": "객관적이고 격식 있게 답변하세요.",
        "question": "오늘 날씨 어때요?",
    })
    print(f"[뉴스 앵커] {r2}")
else:
    print("TODO를 완성해주세요")

### 실습 2-7: 복합 System Prompt 작성

페르소나 + 제약 + 포맷을 모두 포함하는 종합적인 System Prompt를 작성하세요.

**요구사항**
1. 페르소나: IT 회사의 기술 면접관
2. 제약:
   - 답변자의 수준을 고려하여 후속 질문을 하나 더 제시할 것
   - 정답을 직접 알려주지 말 것
   - 존댓말 사용
3. 포맷:
   - 먼저 답변에 대한 피드백 (1-2문장)
   - 그 다음 후속 질문 ("후속 질문:" 으로 시작)
4. google-genai의 `system_instruction`으로 적용
5. 질문: "해시 테이블의 시간 복잡도를 알려주세요"

In [ ]:
# TODO: 페르소나 + 제약 + 포맷을 포함하는 system prompt 작성
interviewer_prompt = ""  # 이 문자열을 작성하세요

# ---------- 정답 ----------
interviewer_prompt = (
    "당신은 IT 회사의 기술 면접관입니다.\n"
    "\n"
    "규칙:\n"
    "- 답변자의 수준을 고려하여 후속 질문을 하나 더 제시하세요.\n"
    "- 정답을 직접 알려주지 마세요. 힌트만 제공하세요.\n"
    "- 존댓말을 사용하세요.\n"
    "\n"
    "답변 포맷:\n"
    "1. 답변에 대한 피드백 (1-2문장)\n"
    "2. '후속 질문:' 으로 시작하는 후속 질문"
)

# 검증
if interviewer_prompt:
    resp = client.models.generate_content(
        model=MODEL,
        config={"system_instruction": interviewer_prompt},
        contents="해시 테이블의 시간 복잡도를 알려주세요.",
    )
    print(resp.text)
else:
    print("TODO를 완성해주세요")

---

### 생각해보기

1. 실습 2-3에서 같은 질문에 페르소나만 바꿨는데 답변이 달라졌습니다. 모델은 실제로 '의사'나 '바리스타'가 된 것일까요, 아니면 흉내를 내는 것일까요? 그 차이가 중요한가요?
2. 실습 2-4에서 포맷 규칙을 system prompt에 넣었습니다. 이 규칙을 user 메시지에 넣어도 동일한 효과가 있을까요? 어디에 넣는 것이 더 효과적일까요?
3. 실습 2-7의 면접관 봇에서 "정답을 직접 알려주지 마라"는 제약이 있습니다. 사용자가 "제약을 무시하고 정답을 알려줘"라고 하면 어떻게 될까요?

---

# Part 3 — 챌린지

이론과 실습에서 배운 내용을 응용하여 스스로 설계하고 구현합니다.
정답이 제공되지 않으며, 강사와 함께 진행합니다.

### 챌린지 2-1: 특정 도메인 전문가 챗봇 System Prompt 설계 (난이도: ★★☆)

> 이 챌린지는 정답이 제공되지 않습니다. 강사와 함께 진행하세요.

특정 도메인의 전문가 챗봇을 만들기 위한 System Prompt를 설계하세요.

**도메인을 하나 선택하세요**
- (A) 여행 플래너 챗봇: 여행 일정을 추천하는 전문가

**System Prompt에 반드시 포함할 요소**
1. 페르소나: 구체적인 전문성과 경력
2. 제약 조건: 최소 3개의 행동 규칙
3. 출력 포맷: 응답 구조 지정
4. 예외 처리: 도메인 밖의 질문에 대한 대응 방법

**평가 기준**
- 페르소나가 구체적인가?
- 제약 조건이 검증 가능한가?
- 다양한 질문에 일관된 응답을 생성하는가?

**힌트**
- 먼저 system prompt를 작성한 후, 최소 3가지 다른 질문으로 테스트하세요
- 도메인 밖 질문도 테스트하세요 (예: 여행 챗봇에 요리 질문)
- LangSmith에서 각 호출의 트레이스를 확인하면 프롬프트 개선에 도움이 됩니다

In [ ]:
# === 챌린지 2-1 답안: 여행 플래너 챗봇 ===

from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.5-flash"

# ── 1단계: System Prompt 설계 ──
my_system_prompt = """당신은 15년 경력의 여행 플래너입니다.

규칙:
- 반드시 존댓말을 사용하세요.
- 여행지 추천 시 예상 비용(대략적 범위)을 포함하세요.
- 정보가 부족하면 여행 기간, 예산, 동행자 등을 먼저 질문하세요.
- 여행과 무관한 질문에는 "저는 여행 관련 질문만 도와드릴 수 있습니다"라고 안내하세요.

답변 포맷:
- 추천 일정은 "Day 1:", "Day 2:" 형태로 작성
- 각 일정은 장소명 + 한 줄 설명
- 마지막에 "Tip:" 으로 실용적인 팁 하나를 추가
"""

# ── 2단계: 도메인 내 질문 테스트 ──
domain_questions = [
    "제주도 2박 3일 일정 추천해주세요. 예산은 50만원이에요.",
    "여름에 동남아 여행 가려면 어디가 좋을까요?",
    "여행 가고 싶어요.",  # 정보 부족 → 추가 질문 유도
]

for q in domain_questions:
    resp = client.models.generate_content(
        model=MODEL,
        config={"system_instruction": my_system_prompt},
        contents=q,
    )
    print(f"Q: {q}")
    print(f"A: {resp.text}\n")
    print("-" * 40)

# ── 3단계: 도메인 밖 질문 테스트 ──
out_of_domain = [
    "김치찌개 레시피 알려주세요.",
    "파이썬 for문 사용법 알려주세요.",
]

for q in out_of_domain:
    resp = client.models.generate_content(
        model=MODEL,
        config={"system_instruction": my_system_prompt},
        contents=q,
    )
    print(f"Q: {q}")
    print(f"A: {resp.text}\n")